In [94]:
import os
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [95]:
from pydantic import BaseModel, Field
from typing import List, Optional

class Equation(BaseModel):
    name: str
    latex: str
    context: str

class Concept(BaseModel):
    topic: str = Field(..., description="The unique name of the concept")
    chunk_type: str = Field(..., description="Type of chunk: 'theory', 'derivation', 'numerical'")
    description: str = Field(..., description="A concise 2-3 sentence technical explanation")
    equations: Optional[List[Equation]] = Field(None, description="LaTeX formatted formulas related to this topic")
    subtopics: Optional[List[str]] = Field(None, description="Specific smaller components or sub-chapters")
    prerequisites: List[str] = Field(..., description="List of topic names that must be understood first")
    difficulty_score: float = Field(..., description="Scale 1-5 of how complex the topic is")
    parent_unit: str = Field(..., description="The main Unit or Chapter this belongs to")

In [96]:
from pymupdf4llm import to_markdown
import pathlib
# pymupdf4llm → Markdown → MarkdownHeaderTextSplitter

doc = to_markdown("../data/edc.pdf")

output_file_path = pathlib.Path("../data/edc.md")
output_file_path.write_bytes(doc.encode("utf-8"))

print(f"Markdown content successfully saved to {output_file_path}")

OCR disabled because Tesseract language data not found.


KeyboardInterrupt: 

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

splitter = MarkdownHeaderTextSplitter(headers_to_split_on=[
    ("##", "section")
])

with open("P:/notebook-lm-mini/data/edc2.md", "r", encoding="utf-8") as f:
    md_text = f.read()

chunks = splitter.split_text(md_text)

In [ ]:
chunks[0]

Document(metadata={'section': '**CHAPTER OBJECTIVES**'}, page_content='●  \n- Learn about constant gain, summing, and buffering amplifiers  \n- Understand how an active filter works  \n- Describe different types of controlled sources')

In [ ]:
last_section = ""
for chunk in chunks:
    if chunk.metadata["section"] != "":
        last_section = chunk.metadata["section"]
    else:
        chunk.metadata["section"] = last_section

In [ ]:
import re

cleaned_chunks = []
for chunk in chunks:
    cleaned = re.sub(r'\*\*==> picture.*?<==\*\*', '', chunk.page_content)
    cleaned = re.sub(r'\*\*----- Start of picture text -----\*\*.*?\*\*----- End of picture text -----\*\*', '', cleaned, flags=re.DOTALL)
    cleaned = cleaned.strip()

    if len(cleaned) < 10:
        continue
    
    chunk.page_content = cleaned
    cleaned_chunks.append(chunk)

print(f"Chunks after cleaning: {len(cleaned_chunks)}")

Chunks after cleaning: 20


In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    max_tokens=500,
    api_key="YOUR_GROQ_API_KEY"
)
structured_llm = llm.with_structured_output(Concept)

prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert educational knowledge graph builder specializing in electrical and electronics engineering.

Extract a single, specific, standalone technical concept from the given textbook chunk.

TOPIC:
- The single most specific technical idea in the chunk
- Must be a precise technical term, max 5 words
- Never use section headings or generic names like "Op-Amp Basics"
- Set to "SKIP" if chunk is introductory, transitional, or has no standalone concept

DESCRIPTION:
- Exactly 2 sentences, hard stop after 2nd sentence
- First sentence: what the concept is and how it works
- Second sentence: why it matters or what it enables
- Must be technical and precise, no filler

DIFFICULTY SCORE:
- 1-2: definitional, no prerequisites
- 3: requires 1-2 prerequisites, moderate abstraction
- 4: multi-prerequisite, involves derivation or circuit analysis
- 5: heavy math, abstract reasoning, cross-topic synthesis

PREREQUISITES:
- Specific technical concept names only
- Only concepts strictly necessary to understand this topic
- Never vague terms like "Op-Amp Basics" or "Circuit Theory"
- Empty list if none

EQUATIONS:
- Only if explicitly present as a formula in the text
- name: actual equation name, never a reference like "Equation 10.3"
- Never hallucinate equations

SUBTOPICS:
- Only specific sub-components explicitly mentioned in this chunk
- Empty list if none

CHUNK TYPE:
- REQUIRED. Must be exactly one of: 'theory', 'derivation', 'numerical'"""),
    ("human", """Section: {section}
Parent Unit: {parent_unit}

Text:
{chunk_content}

Rules:
- Set topic to SKIP if no standalone technical concept exists
- chunk_type is REQUIRED, always include it
- description is exactly 2 sentences, stop after the 2nd sentence
- equation name must be descriptive, never a reference number""")
])


chain = prompt | structured_llm

p:\notebook-lm-mini\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
import time

concepts = []
for chunk in cleaned_chunks:
    try:
        result = chain.invoke({
            "section": chunk.metadata.get("section", ""),
            "parent_unit": "Operational Amplifiers",
            "chunk_content": chunk.page_content
        })
        if result.topic != "SKIP":
            concepts.append(result)
    except Exception as e:
        print(f"Failed chunk: {chunk.metadata.get('section', '')} — {e}")
        continue
    time.sleep(3)

print(f"Valid concepts extracted: {len(concepts)}")

Valid concepts extracted: 18


In [ ]:
chunk_types = []
for c in concepts:
    if c.chunk_type == 'theory':
        chunk_types.append(c.description)

chunk_types = list(set(chunk_types))
print(f"Unique chunk types: {len(chunk_types)}")

Unique chunk types: 17


In [ ]:
i = 0
for chunk in chunk_types:
    print(f"{i + 1}: {chunk}")
    i += 1

1: A voltage buffer circuit is a type of amplifier that provides isolation between an input signal and a load, with unity voltage gain and no phase or polarity inversion, and it has very high input impedance and low output impedance. This circuit is useful for providing multiple outputs from a single input signal without affecting each other, and it is commonly used in instrumentation circuits.
2: A voltage-controlled current source is a circuit that provides an output current controlled by an input voltage, with the output current dependent on the input voltage. This enables the creation of practical circuits, such as the one shown in Fig. 11.19, where the output current through a load resistor can be controlled by the input voltage.
3: An inverting amplifier is a type of op-amp circuit that provides a precise gain or amplification by inverting the input signal, with the resulting gain being determined by the ratio of resistances in the circuit. This type of amplifier is crucial in va

In [ ]:
import json

concepts_data = [c.model_dump() for c in concepts]
with open("concepts.json", "w") as f:
    json.dump(concepts_data, f, indent=2)

print(f"Saved {len(concepts_data)} concepts")

NameError: name 'concepts' is not defined

In [ ]:
from neo4j import GraphDatabase

os.environ["NEO4J_URI"] = "bolt://localhost:7687"
os.environ["NEO4J_USERNAME"] = "neo4j"
os.environ["NEO4J_PASSWORD"] = "pk142145"

driver = GraphDatabase.driver(
    os.environ["NEO4J_URI"],
    auth=(os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"])
)

driver.verify_connectivity()
print("Connected to Neo4j")

Connected to Neo4j


In [ ]:
import json
from pathlib import Path

concepts = []

for json_file in Path(".").glob("concepts.json"):
    with open(json_file, "r") as f:
        data = json.load(f)
    
    for item in data:
        if item.get("equations"):
            item["equations"] = [Equation(**eq) for eq in item["equations"]]
        
        concept = Concept(**item)
        concepts.append(concept)

print(f"Total concepts loaded: {len(concepts)}")

Total concepts loaded: 43


In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8300.61it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:

def create_concept_node(tx, concept: Concept):
    description_embedding = embedding_model.encode(concept.description).tolist()
    topic_embedding = embedding_model.encode(concept.topic).tolist()

    query = """
    MERGE (c:Concept {topic: $topic})
    SET c.description = $description,
        c.difficulty_score = $difficulty_score,
        c.parent_unit = $parent_unit,
        c.chunk_type = $chunk_type,
        c.subtopics = $subtopics,
        c.equations = $equations,
        c.description_embedding = $description_embedding,
        c.topic_embedding = $topic_embedding
    """

    equations = [eq.model_dump() for eq in concept.equations] if concept.equations else []
    
    tx.run(query,
        topic=concept.topic,
        description=concept.description,
        difficulty_score=concept.difficulty_score,
        parent_unit=concept.parent_unit,
        chunk_type=concept.chunk_type,
        subtopics=concept.subtopics or [],
        equations=str(equations),
        description_embedding=description_embedding,
        topic_embedding=topic_embedding
    )

with driver.session() as session:
    for concept in concepts:
        session.execute_write(create_concept_node, concept)

print(f"Nodes created: {len(concepts)}")

Nodes created: 43


In [ ]:
from fuzzywuzzy import process, fuzz

def create_prerequisites(tx, topic, prerequisite):
    query = """
    MATCH (c:Concept {topic: $topic})
    MATCH (p:Concept {topic: $prerequisite})
    MERGE (p)-[:PREREQUISITE_FOR]->(c)
    """
    tx.run(query, topic=topic, prerequisite=prerequisite)

existing_topics = [c.topic for c in concepts]

with driver.session() as session:
    for concept in concepts:
        for prereq in concept.prerequisites:
            resolved = process.extractOne(
                prereq,
                existing_topics,
                scorer=fuzz.token_sort_ratio
            )
            if resolved and resolved[1] >= 70 and resolved[0] != concept.topic:
                session.execute_write(create_prerequisites, concept.topic, resolved[0])

In [ ]:
def create_part_of_relationships(tx, concept):
    query = """
    MERGE (u:Unit {name: $parent_unit})
    MATCH (c:Concept {topic: $topic})
    MERGE (c)-[:PART_OF]->(u)
    """
    tx.run(query,
        parent_unit=concept.parent_unit,
        topic=concept.topic
    )

with driver.session() as session:
    for concept in concepts:
        session.execute_write(create_part_of_relationships, concept)
                
print("PART_OF relationships created")

PART_OF relationships created


## MSMS path finder

In [ ]:
import networkx as nx
from typing import Dict, List, Tuple
from tinydb import TinyDB, Query
from datetime import datetime

student_id = "STUDENT001"

class MSMSPlanner:
    def __init__(self, driver, mastery: Dict[str, float], lam1=0.4, lam2=0.3, lam3=0.2, lam4=0.1):
        self.driver = driver
        self.mastery = mastery
        self.lam1, self.lam2, self.lam3, self.lam4 = lam1, lam2, lam3, lam4
        self.G = self._load_graph()
        self.lam1 = lam1
        self.lam2 = lam2
        self.lam3 = lam3
        self.lam4 = lam4
    
    def _load_graph(self) -> nx.DiGraph:
        G = nx.DiGraph()
        
        with self.driver.session() as session:
            nodes = session.run("""
                MATCH (c:Concept) 
                RETURN c.topic AS topic, 
                    c.difficulty_score AS difficulty, 
                    c.parent_unit AS parent_unit
            """)
            for record in nodes:
                G.add_node(
                    record["topic"],
                    difficulty=record["difficulty"] or 3.0,
                    parent_unit=record["parent_unit"]
                )
            
            edges = session.run("""
                MATCH (p:Concept)-[:PREREQUISITE_FOR]->(c:Concept)
                RETURN p.topic AS source, c.topic AS target
            """)
            for record in edges:
                G.add_edge(record["source"], record["target"])
        
        return G
    
    def _cost(self, u: str, v: str) -> float:
        mastery = self.mastery.get(v, 0.0)
        mastery_cost = 1.0 - mastery

        difficulty = self.G.nodes[v].get("difficulty", 3.0)
        difficulty_cost = (difficulty - 1) / 4.0

        in_degree_cost = self.G.in_degree(v) / max(len(self.G.nodes()), 1)

        out_degree_benefit = self.G.out_degree(v) / max(len(self.G.nodes()), 1)

        cost = (
            self.lam1 * mastery_cost +
            self.lam2 * difficulty_cost +
            self.lam3 * in_degree_cost -
            self.lam4 * out_degree_benefit
        )

        return max(cost, 0.001)
    
    def _get_sources_and_sinks(self, target_topics: List[str], threshold: float = 0.7) -> Tuple[List[str], List[str]]:
        sources = [
            node for node in self.G.nodes()
            if self.mastery.get(node, 0.0) >= threshold
        ]

        if not sources:
            sources = [
                node for node in self.G.nodes()
                if self.G.in_degree(node) == 0
            ]
        
        sinks = [
            topic for topic in target_topics
            if topic in self.G.nodes()
            and self.mastery.get(topic, 0.0) < threshold
        ]
        
        return sources, sinks
    
    def _compute_paths(self, sources: List[str], sinks: List[str]) -> Dict:
        paths = {}
        
        for src in sources:
            paths[src] = {}
            for sink in sinks:
                if src == sink:
                    continue
                try:
                    path = nx.dijkstra_path(
                        self.G, 
                        src, 
                        sink, 
                        weight=lambda u, v, d: self._cost(u, v)
                    )
                    cost = sum(
                        self._cost(path[i], path[i + 1])
                        for i in range(len(path) - 1)
                    )
                    paths[src][sink] = {
                        "path": path,
                        "cost": cost
                    }
                except (nx.NetworkXNoPath, nx.NodeNotFound):
                    pass
        
        return paths
    
    def _greedy_set_cover(self, all_paths: Dict, sinks: List[str]) -> List[List[str]]:
        candidates = []
        for src, destinations in all_paths.items():
            for sink, data in destinations.items():
                path = data["path"]
                cost = data["cost"]
                covered = set(path) & set(sinks)
                candidates.append({
                    "path": path,
                    "cost": cost,
                    "covered": covered
                })
        
        uncovered = set(sinks)
        selected = []
        
        while uncovered:
            best = max(
                (c for c in candidates if c["covered"] & uncovered),
                key=lambda c: (len(c["covered"] & uncovered), -c["cost"]),
                default=None
            )
            
            if not best:
                break
            
            selected.append(best["path"])
            uncovered -= best["covered"]
        
        return selected


## Student Manager

In [ ]:
from tinydb import TinyDB, Query
from datetime import datetime
from typing import Dict, List, Optional

class StudentStateManager:
    def __init__(self, student_id: str, db_path: str = "student_sessions.json"):
        self.student_id = student_id
        self.db = TinyDB(db_path)
        self.Student = Query()
        self._ensure_student_exists()
    
    def _ensure_student_exists(self):
        result = self.db.search(self.Student.student_id == self.student_id)
        if not result:
            self.db.insert({
                "student_id": self.student_id,
                "mastery": {},
                "planned_path": [],
                "progress": {},
                "created_at": datetime.now().isoformat(),
                "last_updated": datetime.now().isoformat()
            })
    
    def get_mastery(self) -> Dict[str, float]:
        result = self.db.search(self.Student.student_id == self.student_id)
        return result[0].get("mastery", {})
    
    def update_mastery(self, topic: str, new_score: float):
        result = self.db.search(self.Student.student_id == self.student_id)
        mastery = result[0].get("mastery", {})
        mastery[topic] = round(max(0.0, min(1.0, new_score)), 2)
        self.db.update(
            {
                "mastery": mastery,
                "last_updated": datetime.now().isoformat()
            },
            self.Student.student_id == self.student_id
        )
    
    def save_planned_path(self, paths: List[List[str]]):
        self.db.update(
            {
                "planned_path": paths,
                "last_updated": datetime.now().isoformat()
            },
            self.Student.student_id == self.student_id
        )
    
    def get_planned_path(self) -> List[List[str]]:
        result = self.db.search(self.Student.student_id == self.student_id)
        return result[0].get("planned_path", [])
    
    def update_progress(self, topic: str, score: float):
        """
        score: 0.0 to 1.0 representing answer quality
        1.0 = perfect, 0.5 = partial, 0.0 = wrong
        """
        result = self.db.search(self.Student.student_id == self.student_id)
        progress = result[0].get("progress", {})
        
        if topic not in progress:
            progress[topic] = {
                "attempts": 0,
                "cumulative_score": 0.0
            }
        
        progress[topic]["attempts"] += 1
        progress[topic]["cumulative_score"] += score
        
        new_mastery = (
            progress[topic]["cumulative_score"] / 
            progress[topic]["attempts"]
        )
        
        self.db.update(
            {
                "progress": progress,
                "last_updated": datetime.now().isoformat()
            },
            self.Student.student_id == self.student_id
        )
        self.update_mastery(topic, new_mastery)
    
    def get_progress(self) -> Dict:
        result = self.db.search(self.Student.student_id == self.student_id)
        return result[0].get("progress", {})
    
    def get_next_concept(self) -> Optional[str]:
        planned_path = self.get_planned_path()
        mastery = self.get_mastery()
        
        for path in planned_path:
            for concept in path:
                if mastery.get(concept, 0.0) < 0.7:
                    return concept
        return None

## AgentState

In [127]:
class GraphState(TypedDict):
    student_id: str
    messages: List[dict]
    current_input: str
    target_topics: List[str]
    known_topics: List[str]
    current_concept: str
    current_question: str
    student_answer: str
    answer_score: float
    diagnosis_report: dict
    planned_paths: List[List[str]]
    current_path_index: int
    current_concept_index: int
    final_response: str
    turn: str

## AGENT

In [ ]:
from pydantic import BaseModel
from typing import List

class IntentOutput(BaseModel):
    known_topics: List[str]
    target_topics: List[str]

In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
from typing import List, Optional
import numpy as np

class IntentParser:
    
    def __init__(self, driver):
        self.driver = driver
        self.biencoder = SentenceTransformer('all-MiniLM-L6-v2')
        self.crossencoder = CrossEncoder('cross-encoder/stsb-roberta-base')
        self.topics, self.topic_embeddings = self._load_node_embeddings()
    
    def _load_node_embeddings(self):
        with self.driver.session() as session:
            result = session.run("""
                MATCH (c:Concept)
                RETURN c.topic AS topic, c.topic_embedding AS embedding
            """)
            topics, embeddings = [], []
            for record in result:
                topics.append(record["topic"])
                embeddings.append(record["embedding"])
        return topics, np.array(embeddings)
    
    def _extract_intent(self, student_prompt: str) -> IntentOutput:
        llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
        structured_llm = llm.with_structured_output(IntentOutput)
        
        prompt = ChatPromptTemplate.from_messages([
            ("system", """You are an intent extractor for a learning system.

            Extract TWO things from the student message:
            1. known_topics: ONLY concepts the student EXPLICITLY says they know
            2. target_topics: ONLY concepts the student EXPLICITLY says they want to learn

            Rules:
            - Never extract vague terms like "basic amplifiers" or "circuits"
            - Always extract the most specific technical concept name possible
            - known_topics must be EXPLICITLY stated as known — never infer
            - target_topics must be EXPLICITLY stated as goals — never infer
            - "from scratch" or "everything" → return empty lists for both
            - "I don't understand X" → target: [X], known: []
            - Maximum 3 topics per list

            Examples:
            - "I know basic amplifiers" → known: ["differential amplifier", "inverting amplifier"], target: []
            - "I want to learn filters" → known: [], target: ["low pass filter"]
            - "I don't understand virtual ground" → known: [], target: ["virtual ground"]
            - "from scratch" → known: [], target: []

            Return empty lists only if truly not mentioned."""),
            ("human", "{student_prompt}")
        ])
        
        chain = prompt | structured_llm
        return chain.invoke({"student_prompt": student_prompt})
    
    def _retrieve_candidates(self, query: str, top_k: int = 10) -> List[str]:
        query_embedding = self.biencoder.encode([query])
        similarities = cosine_similarity(query_embedding, self.topic_embeddings)[0]
        top_k_indices = np.argsort(similarities)[-top_k:][::-1]
        return [self.topics[i] for i in top_k_indices]
    
    def _rerank_candidates(self, query: str, candidates: List[str], threshold: float = 0.0) -> Optional[str]:
        pairs = [[query, candidate] for candidate in candidates]
        scores = self.crossencoder.predict(pairs)
        best_idx = np.argmax(scores)
        best_score = scores[best_idx]
        
        if best_score < threshold:
            return None
        
        return candidates[best_idx]
    
    def parse(self, student_prompt: str) -> IntentOutput:
        raw_intent = self._extract_intent(student_prompt)
        
        resolved_known = []
        for topic in raw_intent.known_topics:
            candidates = self._retrieve_candidates(topic)
            resolved = self._rerank_candidates(topic, candidates)
            if resolved is not None:
                resolved_known.append(resolved)
        
        resolved_targets = []
        for topic in raw_intent.target_topics:
            candidates = self._retrieve_candidates(topic)
            resolved = self._rerank_candidates(topic, candidates)
            if resolved is not None:
                resolved_targets.append(resolved)
        
        return IntentOutput(
            known_topics=resolved_known,
            target_topics=resolved_targets
        )

In [99]:
parser = IntentParser(driver)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11879.27it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 7785.09it/s]
RobertaForSequenceClassification LOAD REPORT from: cross-encoder/stsb-roberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [100]:
result = parser.parse("I know basic amplifiers, I want to learn integrators and slew rate")
print("Test 1:", result)

result = parser.parse("I don't understand how feedback works in op-amps")
print("Test 2:", result)

result = parser.parse("I want to learn everything about op-amps from scratch")
print("Test 3:", result)

Test 1: known_topics=['Differential Amplifier', 'Inverting Amplifier'] target_topics=['Integrator Circuit', 'Slew Rate']
Test 2: known_topics=[] target_topics=['Op-Amp Gain']
Test 3: known_topics=[] target_topics=[]


In [101]:
query = "basic amplifiers"
candidates = parser._retrieve_candidates(query, top_k=10)
pairs = [[query, c] for c in candidates]
scores = parser.crossencoder.predict(pairs)
for c, s in zip(candidates, scores):
    print(f"  {c}: {s:.4f}")

  Summing Amplifier: 0.4737
  Instrumentation Amplifier: 0.5090
  Noninverting Amplifier: 0.4391
  Inverting Amplifier: 0.4016
  Differential Amplifier: 0.3642
  Inverting Amplifier Gain: 0.4172
  CMOS Differential Amplifier: 0.3260
  Op-Amp Gain: 0.4568
  Common-Mode Gain: 0.3752
  Instrumentation Circuits: 0.3145


In [102]:

def planner_agent(state: GraphState) -> GraphState:
    student_id = state["student_id"]
    known_topics = state["known_topics"]
    target_topics = state["target_topics"]
    
    state_manager = StudentStateManager(student_id)
    mastery = state_manager.get_mastery()

    for topic in known_topics:
        if topic not in mastery:
            mastery[topic] = 0.7
    
    optimizer = MSMSPlanner(driver, mastery)
    sources, sinks = optimizer._get_sources_and_sinks(target_topics)
    all_paths = optimizer._compute_paths(sources, sinks)
    planned_paths = optimizer._greedy_set_cover(all_paths, sinks)
    
    state_manager.save_planned_path(planned_paths)
    
    state["planned_paths"] = planned_paths
    state["current_concept"] = planned_paths[0][0] if planned_paths else ""
    
    return state

In [143]:
test_state = GraphState(
    student_id="newUser",
    messages=[],
    current_input="I know basic amplifiers, I want to learn integrators and slew rate",
    target_topics=["Integrator Circuit", "Slew Rate"],
    known_topics=["Differential Amplifier"],
    current_concept="",
    current_question="",
    student_answer="",
    answer_score=0.0,
    diagnosis_report={},
    planned_paths=[],
    current_path_index=0,
    current_concept_index=0,
    final_response=""
)

result = planner_agent(test_state)
print(f"Planned paths: {result['planned_paths']}")
print(f"First concept to learn: {result['current_concept']}")

Planned paths: [['Differential Amplifier', 'Op-Amp Gain', 'Inverting Amplifier', 'Slew Rate'], ['Differential Amplifier', 'Op-Amp Gain', 'Inverting Amplifier', 'Integrator Circuit']]
First concept to learn: Differential Amplifier


## Diagnoser 

In [104]:
class Question(BaseModel):
    concept: str
    question_text: str
    question_type: str 
    expected_answer: str
    
class EvaluationResult(BaseModel):
    score: float 
    feedback: str 
    misconceptions: List[str] 

In [ ]:
class DiagnoserAgent:
    
    def __init__(self, driver):
        self.driver = driver
        self.llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.3, max_tokens=200)
    
    def _fetch_concept(self, topic: str) -> dict:
        with self.driver.session() as session:
            result = session.run("""
                MATCH (c:Concept {topic: $topic})
                RETURN c.description AS description,
                       c.equations AS equations,
                       c.difficulty_score AS difficulty_score,
                       c.chunk_type AS chunk_type,
                       c.subtopics AS subtopics
            """, topic=topic)
            record = result.single()
            return dict(record) if record else {}
    
    def generate_question(self, topic: str) -> Question:
        concept = self._fetch_concept(topic)
        
        structured_llm = self.llm.with_structured_output(Question)
        
        prompt = ChatPromptTemplate.from_messages([
            ("system", """You are an expert engineering tutor generating assessment questions.

            Generate ONE question for the given concept based on its type:
            - theory → conceptual question testing understanding of how/why
            - derivation → equation-based question requiring mathematical reasoning  
            - numerical → problem-solving question with specific values

            Rules:
            - question must be specific to the concept, not generic
            - expected_answer must be a complete reference answer
            - question_type must match chunk_type"""),
                        ("human", """Concept: {topic}
            Description: {description}
            Chunk Type: {chunk_type}
            Difficulty: {difficulty_score}
            Equations: {equations}
            Subtopics: {subtopics}

            Generate a question that tests deep understanding of this concept.""")
        ])
        
        chain = prompt | structured_llm
        return chain.invoke({
            "topic": topic,
            "description": concept.get("description", ""),
            "chunk_type": concept.get("chunk_type", "theory"),
            "difficulty_score": concept.get("difficulty_score", 3.0),
            "equations": concept.get("equations", ""),
            "subtopics": concept.get("subtopics", [])
        })
    
    def evaluate_answer(self, question: Question, student_answer: str) -> EvaluationResult:
        structured_llm = self.llm.with_structured_output(EvaluationResult)
        
        prompt = ChatPromptTemplate.from_messages([
            ("system", """You are an expert engineering tutor evaluating a student's answer.

            Score the answer from 0.0 to 1.0:
            - 1.0: complete, accurate, shows deep understanding
            - 0.7-0.9: mostly correct with minor gaps
            - 0.4-0.6: partial understanding, significant gaps
            - 0.1-0.3: minimal understanding, major misconceptions
            - 0.0: completely wrong or no attempt

            Rules:
            - feedback must be specific and constructive
            - misconceptions must identify exact conceptual gaps
            - be strict but fair"""),
                        ("human", """Concept: {concept}
            Question: {question_text}
            Expected Answer: {expected_answer}
            Student Answer: {student_answer}

            Evaluate the student's answer.""")
        ])
        
        chain = prompt | structured_llm
        return chain.invoke({
            "concept": question.concept,
            "question_text": question.question_text,
            "expected_answer": question.expected_answer,
            "student_answer": student_answer
        })
    
    def run(self, state: GraphState) -> GraphState:
        topic = state["current_concept"]

        if not state["current_question"]:
            question = self.generate_question(topic)
            state["current_question"] = question.question_text
            state["diagnosis_report"] = {
                "concept": question.concept,
                "question_text": question.question_text,
                "expected_answer": question.expected_answer,
                "question_type": question.question_type
            }
            return state
        
        if state["student_answer"]:
            question = Question(**state["diagnosis_report"])
            evaluation = self.evaluate_answer(question, state["student_answer"])
            

            state_manager = StudentStateManager(state["student_id"])
            state_manager.update_progress(topic, evaluation.score)

            state["answer_score"] = evaluation.score
            state["diagnosis_report"]["feedback"] = evaluation.feedback
            state["diagnosis_report"]["misconceptions"] = evaluation.misconceptions
            
            if evaluation.score >= 0.7:
                state["current_concept_index"] += 1
                current_path = state["planned_paths"][state["current_path_index"]]
                if state["current_concept_index"] < len(current_path):
                    state["current_concept"] = current_path[state["current_concept_index"]]
                state["current_question"] = ""
                state["student_answer"] = ""
        
        return state

In [135]:
diagnoser = DiagnoserAgent(driver)

question = diagnoser.generate_question("Integrator Circuit")
print(f"Question: {question.question_text}")
print(f"Type: {question.question_type}")
print(f"Expected: {question.expected_answer}")

Question: What is the fundamental principle behind the operation of an integrator circuit, and how does it enable the solution of differential equations?
Type: theory
Expected: The integrator circuit uses a capacitor as the feedback component, allowing it to integrate the input signal over time, enabling the solution of differential equations and providing the ability to electrically solve analogs of physical system operation.


In [109]:
evaluation = diagnoser.evaluate_answer(question, 
    "An integrator uses a capacitor in the feedback loop. When a voltage is applied, the capacitor charges over time producing a ramp output. The output is the integral of the input scaled by 1/RC. This allows modeling of physical systems by converting differential equations into electrical circuits.")

print(f"Score: {evaluation.score}")
print(f"Feedback: {evaluation.feedback}")
print(f"Misconceptions: {evaluation.misconceptions}")

evaluation2 = diagnoser.evaluate_answer(question,
    "Integrator uses a capacitor to store charge and produces an output voltage.")

print(f"\nScore: {evaluation2.score}")
print(f"Feedback: {evaluation2.feedback}")
print(f"Misconceptions: {evaluation2.misconceptions}")

Score: 0.9
Feedback: The student's answer is mostly correct, but it lacks a clear explanation of how the integrator circuit enables the solution of differential equations. The student correctly identifies the use of a capacitor in the feedback loop and the resulting ramp output, but could improve by explicitly stating how this allows for the electrical solution of analogs of physical system operation.
Misconceptions: ['Lack of explicit connection to solving differential equations', 'No mention of the role of the integrator in modeling physical systems beyond converting differential equations into electrical circuits']

Score: 0.4
Feedback: The student's answer is partially correct, but it lacks the depth and clarity required to fully understand the fundamental principle behind the operation of an integrator circuit. The answer should have mentioned the integration of the input signal over time and its relation to solving differential equations. To improve, the student should focus on u

In [ ]:
from langgraph.graph import END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import MemorySaver

def route_turn(state: GraphState) -> str:
    return state["turn"]
    
def intent_parser_node(state: GraphState) -> GraphState:
    parser = IntentParser(driver)
    result = parser.parse(state["current_input"])
    state["known_topics"] = result.known_topics
    state["target_topics"] = result.target_topics
    state["turn"] = "plan"
    return state

def planner_node(state: GraphState) -> GraphState:
    state_manager = StudentStateManager(state["student_id"])
    mastery = state_manager.get_mastery()
    
    for topic in state["known_topics"]:
        if topic not in mastery:
            mastery[topic] = 0.9
    
    optimizer = MSMSPlanner(driver, mastery)
    sources, sinks = optimizer._get_sources_and_sinks(state["target_topics"])
    all_paths = optimizer._compute_paths(sources, sinks)
    planned_paths = optimizer._greedy_set_cover(all_paths, sinks)
    
    state_manager.save_planned_path(planned_paths)
    
    state["planned_paths"] = planned_paths
    state["current_concept"] = planned_paths[0][0] if planned_paths else ""
    state["current_path_index"] = 0
    state["current_concept_index"] = 0
    state["turn"] = "ask"
    return state

def diagnoser_generate_node(state: GraphState) -> GraphState:
    diagnoser = DiagnoserAgent(driver)
    question = diagnoser.generate_question(state["current_concept"])
    state["current_question"] = question.question_text
    state["diagnosis_report"] = {
        "concept": question.concept,
        "question_text": question.question_text,
        "expected_answer": question.expected_answer,
        "question_type": question.question_type
    }
    state["turn"] = "student"
    return state

def human_node(state: GraphState) -> GraphState:
    student_answer = interrupt(state["current_question"])

    if not student_answer or student_answer.strip() == "":
        state["student_answer"] = "__empty__"
    else:
        state["student_answer"] = student_answer.strip()
    
    state["turn"] = "evaluate"
    return state

def diagnoser_evaluate_node(state: GraphState) -> GraphState:
    if state["student_answer"] == "__empty__":
        state["answer_score"] = 0.0
        state["diagnosis_report"]["feedback"] = "No answer provided."
        state["diagnosis_report"]["misconceptions"] = ["Student did not attempt the question"]
    else:
        diagnoser = DiagnoserAgent(driver)
        question = Question(**state["diagnosis_report"])
        evaluation = diagnoser.evaluate_answer(question, state["student_answer"])
        
        state["answer_score"] = evaluation.score
        state["diagnosis_report"]["feedback"] = evaluation.feedback
        state["diagnosis_report"]["misconceptions"] = evaluation.misconceptions
        
        state_manager = StudentStateManager(state["student_id"])
        state_manager.update_progress(state["current_concept"], evaluation.score)
    
    attempts = state["diagnosis_report"].get("attempts", 0) + 1
    state["diagnosis_report"]["attempts"] = attempts
    
    if evaluation.score >= 0.7 or attempts >= 3:
        state["current_concept_index"] += 1
        current_path = state["planned_paths"][state["current_path_index"]]
        if state["current_concept_index"] < len(current_path):
            state["current_concept"] = current_path[state["current_concept_index"]]
        state["current_question"] = ""
        state["student_answer"] = ""
        state["diagnosis_report"]["attempts"] = 0
        state["turn"] = "next"
    else:
        state["turn"] = "respond"
    
    return state

def tutor_respond_node(state: GraphState) -> GraphState:
    llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.3, max_tokens=150)
    
    if state["turn"] == "next":
        current_path = state["planned_paths"][state["current_path_index"]]
        
        if state["current_concept_index"] >= len(current_path):
            if state["current_path_index"] + 1 < len(state["planned_paths"]):
                state["current_path_index"] += 1
                state["current_concept_index"] = 0
                state["current_concept"] = state["planned_paths"][state["current_path_index"]][0]
                state["turn"] = "ask"
            else:
                state["final_response"] = "Congratulations! You have completed all planned learning paths."
                state["turn"] = "end"
                return state
        else:
            state["turn"] = "ask"
        state["final_response"] = f"""✓ Good answer! {state['diagnosis_report'].get('feedback', '')}
        Moving to next concept: **{state['current_concept']}**"""
    
    else:
        misconceptions = state['diagnosis_report'].get('misconceptions', [])
        misconceptions_text = '\n'.join([f"- {m}" for m in misconceptions])
        
        prompt = ChatPromptTemplate.from_messages([
            ("system", """You are an expert engineering tutor. 
        Give a SHORT explanation in maximum 3 sentences.
        Address only the specific misconceptions listed.
        No bullet points, no headers, just plain concise text."""),
            ("human", """Concept: {concept}
        Misconceptions: {misconceptions}

        Give a 3 sentence explanation addressing these misconceptions only.""")
        ])
        
        chain = prompt | llm
        explanation = chain.invoke({
            "concept": state["current_concept"],
            "description": state["diagnosis_report"].get("expected_answer", ""),
            "misconceptions": misconceptions_text
        })
        
        state["final_response"] = f"""Let me clarify **{state['current_concept']}**:

{explanation.content}

Let's try again:
{state['current_question']}"""
        state["turn"] = "ask"
    
    return state

In [140]:
workflow = StateGraph(GraphState)

workflow.add_node("intent_parser", intent_parser_node)
workflow.add_node("planner", planner_node)
workflow.add_node("diagnoser_generate", diagnoser_generate_node)
workflow.add_node("human", human_node)
workflow.add_node("diagnoser_evaluate", diagnoser_evaluate_node)
workflow.add_node("tutor_respond", tutor_respond_node)

workflow.set_entry_point("intent_parser")
workflow.add_edge("intent_parser", "planner")
workflow.add_edge("planner", "diagnoser_generate")
workflow.add_edge("diagnoser_generate", "human")
workflow.add_edge("human", "diagnoser_evaluate")
workflow.add_edge("diagnoser_evaluate", "tutor_respond")

workflow.add_conditional_edges(
    "tutor_respond",
    route_turn,
    {
        "ask": "diagnoser_generate",
        "end": END
    }
)

checkpointer = MemorySaver()
app = workflow.compile(checkpointer=checkpointer)

In [ ]:
config = {"configurable": {"thread_id": "newUser"}}

initial_state = GraphState(
    student_id="newUser",
    messages=[],
    current_input="I know basic amplifiers, I want to learn integrators and slew rate",
    target_topics=[],
    known_topics=[],
    current_concept="",
    current_question="",
    student_answer="",
    answer_score=0.0,
    diagnosis_report={},
    planned_paths=[],
    current_path_index=0,
    current_concept_index=0,
    final_response="",
    turn="start"
)

result = app.invoke(initial_state, config=config)
print(f"\nTutor: {result['final_response']}")
print(f"\nQuestion: {result['current_question']}")

while True:
    student_input = input("\nYou: ")
    result = app.invoke(Command(resume=student_input), config=config)
    print(f"\nTutor: {result['final_response']}")
    
    if result["turn"] == "end":
        print("\nSession complete!")
        break

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4416.46it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6644.77it/s]
RobertaForSequenceClassification LOAD REPORT from: cross-encoder/stsb-roberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Tutor: 

Question: What is the primary function of a differential amplifier and how does it enable the creation of various electronic circuits?

Tutor: Let me clarify **Differential Amplifier**:

A differential amplifier's primary function is to amplify the difference between two input signals, allowing it to reject common-mode noise and provide a high degree of precision. This unique ability makes differential amplifiers essential in various electronic circuits, such as audio equipment, medical devices, and instrumentation systems. The applications of differential amplifiers are diverse, ranging from signal processing and data acquisition to telecommunications and computer networks, where they play a crucial role in enhancing signal quality and reliability.

Let's try again:
What is the primary function of a differential amplifier and how does it enable the creation of various electronic circuits?


UnboundLocalError: cannot access local variable 'evaluation' where it is not associated with a value